In [54]:
import pandas as pd
import numpy as np
import re

In [55]:
train = pd.read_csv("../data/raw/twitter_training.csv", header = None, names = ["tweet_id", "entity", "sentiment", "tweet"])
test = pd.read_csv("../data/raw/twitter_validation.csv", header = None, names = ["tweet_id", "entity", "sentiment", "tweet"])

In [56]:
train.head()

,tweet_id,entity,sentiment,tweet
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [57]:
test.head()

,tweet_id,entity,sentiment,tweet
0,3364,Facebook,Irrelevant,I mentioned on Facebook that I was struggling ...
1,352,Amazon,Neutral,BBC News - Amazon boss Jeff Bezos rejects clai...
2,8312,Microsoft,Negative,@Microsoft Why do I pay for WORD when it funct...
3,4371,CS-GO,Negative,"CSGO matchmaking is so full of closet hacking,..."
4,4433,Google,Neutral,Now the President is slapping Americans in the...


In [58]:
print("Training Dataset shape:", train.shape)
print("Validation Dataset shape:", test.shape)

Training Dataset shape: (74682, 4)
Validation Dataset shape: (1000, 4)


In [59]:
df = pd.concat([train, test], axis = 0, ignore_index = True)

In [60]:
df.shape

(75682, 4)

In [61]:
df = df.drop(["tweet_id", "entity"], axis = 1)

In [62]:
df["sentiment"].value_counts()

sentiment
Negative      22808
Positive      21109
Neutral       18603
Irrelevant    13162
Name: count, dtype: int64

In [63]:
df = df[df["sentiment"] != "Irrelevant"].reset_index(drop = True)

In [64]:
df.shape

(62520, 2)

In [65]:
df.tail()

,sentiment,tweet
62515,Negative,Please explain how this is possible! How can t...
62516,Positive,Good on Sony. As much as I want to see the new...
62517,Positive,Today sucked so it’s time to drink wine n play...
62518,Positive,Bought a fraction of Microsoft today. Small wins.
62519,Neutral,Johnson & Johnson to stop selling talc baby po...


In [66]:
df.isna().sum()

sentiment      0
tweet        571
dtype: int64

In [68]:
df = df.dropna(axis = 0)

In [69]:
df.shape

(61949, 2)

In [70]:
df = df.drop_duplicates(subset=['tweet']).reset_index(drop=True)

In [71]:
df.shape

(57693, 2)

In [72]:
df.tail()

,sentiment,tweet
57688,Negative,I have noticed streamers I watch who are now p...
57689,Neutral,@6th__man playing red dead redemption-\n\n“Oh ...
57690,Neutral,♥️ Suikoden 2\n1️⃣ Alex Kidd in Miracle World\...
57691,Positive,Thank you to Matching funds Home Depot RW paym...
57692,Neutral,Late night stream with the boys! Come watch so...


In [73]:
def clean_tweet(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+',  '', text)  # URLs
    text = re.sub(r'@\w+',            '', text)  # mentions
    text = re.sub(r'#\w+',            '', text)  # hashtags
    text = re.sub(r'[^a-zA-Z\s]',     '', text)  # punctuation
    text = re.sub(r'\s+',             ' ', text)  # extra spaces
    return text.strip()

In [74]:
df['cleaned_tweet'] = df['tweet'].apply(clean_tweet)

In [75]:
df = df[df['cleaned_tweet'].str.len() > 0].reset_index(drop=True)

In [76]:
df = df[['cleaned_tweet', 'sentiment']]

In [77]:
print("Shape:", df.shape)
print(df['sentiment'].value_counts())

Shape: (57589, 2)
sentiment
Negative    21228
Positive    19186
Neutral     17175
Name: count, dtype: int64


In [78]:
from sklearn.preprocessing import LabelEncoder

In [79]:
le = LabelEncoder()
le.fit(df["sentiment"])
df["sentiment"] = le.transform(df["sentiment"])

In [83]:
import tensorflow
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [84]:
tokenizer = Tokenizer(num_words = 10000, oov_token = "<OOV>")

tokenizer.fit_on_texts(df["cleaned_tweet"])

In [85]:
df["tokenized_tweet"] = tokenizer.texts_to_sequences(df["cleaned_tweet"])

In [87]:
padded_seq = pad_sequences(df["tokenized_tweet"], maxlen = 100, padding = "post")

In [88]:
padded_seq.shape

(57589, 100)

In [90]:
sentiment = df["sentiment"]
sentiment.shape

(57589,)

In [89]:
# Saving the models
import pickle

with open("../models/le.pkl", "wb") as f:
    pickle.dump(le, f)

with open("../models/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [93]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(padded_seq, sentiment, test_size = 0.2, random_state = 44, stratify = sentiment)

In [94]:
# Saving the processed data

with open("../data/processed/X_train.pkl", "wb") as f:
    pickle.dump(X_train, f)

with open("../data/processed/X_test.pkl", "wb") as f:
    pickle.dump(X_test, f)

with open("../data/processed/y_train.pkl", "wb") as f:
    pickle.dump(y_train, f)

with open("../data/processed/y_test.pkl", "wb") as f:
    pickle.dump(y_test, f)